# Chatting with a Wikipedia Article Using Retrieval-Augmented Generation
### Keaton Ludwig

## Chatting with a Wikipedia Article using RAG and Hugging Face

### Description

This week's reading built a chatbot that answers questions about a PDF using Retrieval-Augmented Generation (RAG). This notebook does the same thing with a Wikipedia page, scraped with Beautiful Soup. The assignment says the input is just text no matter where it comes from. That's mostly true. The one place it wasn't true was getting clean text out of the HTML in the first place (see Step 1).

## Setup

In [1]:
!pip -q install beautifulsoup4 langchain-text-splitters sentence-transformers faiss-cpu gradio accelerate

In [2]:
import re
import time
import textwrap

import requests
import numpy as np
import pandas as pd
import torch
import faiss
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- configuration -------------------------------------------------------
WIKI_TITLE    = "Nebraska Cornhuskers football"                       # any Wikipedia article title
EMBED_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"     # retriever
LLM_MODEL     = "Qwen/Qwen2.5-1.5B-Instruct"                 # generator
CHUNK_SIZE    = 800      # characters
CHUNK_OVERLAP = 150      # characters
TOP_K         = 4        # chunks handed to the LLM per question

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
if device == "cpu":
    print("No GPU detected - Runtime > Change runtime type > T4 GPU will make generation much faster.")

Running on: cuda


In [3]:
SKIP_SECTIONS = {
    "see also", "references", "notes", "external links", "further reading",
    "bibliography", "sources", "citations", "footnotes", "works cited",
}

JUNK_SELECTORS = [
    "table", "style", "script", "figure", "sup.reference", ".mw-editsection",
    ".hatnote", ".navbox", ".reflist", ".references", ".thumb", ".infobox",
    ".sidebar", ".metadata", ".toc", ".shortdescription", ".mw-empty-elt",
    ".noprint", ".gallery",
]


def clean_text(text):
    text = re.sub(r"\[(?:\d+|[a-z]|citation needed|note \d+)\]", "", text)   # leftover citation markers
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?)])", r"\1", text)    # no space before punctuation
    return text.strip()


def fetch_wikipedia_sections(title):
    url = "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")
    headers = {"User-Agent": "wikipedia-rag-chatbot/1.0 (educational use)"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    page_title = soup.find("h1").get_text(strip=True)
    content = soup.select_one("#mw-content-text > div.mw-parser-output")   # the article body, not the small widgets that reuse this class

    for selector in JUNK_SELECTORS:
        for element in content.select(selector):
            element.decompose()

    sections = []
    h2, h3 = "Introduction", None
    skipping = False
    buffer = []

    def flush():
        text = "\n\n".join(buffer).strip()
        if text and not skipping:
            sections.append({"section": h2 if h3 is None else f"{h2} > {h3}", "text": text})
        buffer.clear()

    for element in content.find_all(["h2", "h3", "p", "ul", "ol", "blockquote"]):
        # nested elements are already covered by their parent list item / quote
        if element.name not in ("h2", "h3") and element.find_parent(["li", "blockquote"]):
            continue

        if element.name == "h2":
            flush()
            h2, h3 = element.get_text(strip=True), None
            skipping = h2.lower() in SKIP_SECTIONS
        elif element.name == "h3":
            flush()
            h3 = element.get_text(strip=True)
        elif element.name in ("ul", "ol"):
            items = [clean_text(li.get_text(" ")) for li in element.find_all("li", recursive=False)]
            buffer.append("\n".join(f"- {item}" for item in items if item))
        else:
            text = clean_text(element.get_text())
            if text:
                buffer.append(text)
    flush()
    return page_title, url, sections

In [4]:
page_title, page_url, sections = fetch_wikipedia_sections(WIKI_TITLE)

total_chars = sum(len(s["text"]) for s in sections)
print(f"Article : {page_title}")
print(f"URL     : {page_url}")
print(f"Sections: {len(sections)}   |   characters of clean text: {total_chars:,}\n")

for s in sections:
    print(f"{len(s['text']):>6,} chars  {s['section']}")

print("\n--- preview of the introduction ---")
print(textwrap.fill(sections[0]["text"][:700], 100) + " ...")

Article : Nebraska Cornhuskers football
URL     : https://en.wikipedia.org/wiki/Nebraska_Cornhuskers_football
Sections: 29   |   characters of clean text: 41,668

 1,681 chars  Introduction
   351 chars  Conference affiliations
 2,741 chars  Head coaches
 3,836 chars  Championships > National championships
 1,265 chars  Championships > Conference championships
   393 chars  Championships > Division championships
 2,265 chars  Bowl games
 1,188 chars  Venues > Nebraska Field
 1,427 chars  Venues > Memorial Stadium
 1,272 chars  Traditions > Gameday traditions
   903 chars  Traditions > Walk-on program
 4,000 chars  Traditions > Uniform history
 3,490 chars  Sellout streak
 1,634 chars  Rivalries > Colorado
 1,194 chars  Rivalries > Iowa
   745 chars  Rivalries > Iowa State
   688 chars  Rivalries > Kansas
 1,181 chars  Rivalries > Kansas State
 1,252 chars  Rivalries > Miami (FL)
 1,197 chars  Rivalries > Minnesota
 1,271 chars  Rivalries > Missouri
 2,976 chars  Rivalries > Oklahoma
  

### Commentary

The section list lines up with the article's table of contents, from the introduction through "Future schedule," and none of the back matter made it in. No References, no External links, no See also. The preview is clean too. The citation numbers and "[edit]" links are gone, and the text reads the way the article does on the page.

Two things weren't perfect. A couple of "Co-champion" lines survived in the championship sections. They're footnotes for tables the scraper threw away, so on their own they mean nothing. And the tables themselves are gone, so any fact that only appears in one (the year-by-year championship listings, for example) can't be found. For this assignment I think that trade was worth it, because the prose covers most of what someone would ask. It would hurt if someone wanted a specific season's record.

The page also has no standalone history section. The program's history is spread across Head coaches, Championships and Sellout streak, so a broad question like "what happened in the 1980s?" has to be stitched together from several places. That's harder for a 4-chunk retrieval than one tidy history paragraph would be.

41,668 characters became 79 chunks, with a median length of 587 characters. That's a small index, but it also means each question only gets to see about 5% of the article.

## Step 2: Chunking

The embedding model squeezes a whole chunk into a single vector, so chunk size is a balancing act. Make chunks too big and each one covers several topics, so its vector turns into a muddy average that doesn't match anything well. There's also a hard ceiling: `all-MiniLM-L6-v2` reads the first 256 tokens and silently drops the rest. Make them too small and you get fragments like "it collapsed" with nothing saying what collapsed.

800 characters is about 150 to 200 tokens, roughly one solid paragraph. That stays under the model's limit with room to spare. The 150-character overlap is there so a fact that lands on a boundary still shows up whole in at least one chunk.

Chunks never cross section boundaries. A chunk that starts in the Iowa rivalry and ends in Iowa State would confuse both retrieval and the model.

Each chunk also gets the article and section name stuck on the front before it's embedded, like `Nebraska Cornhuskers football > Rivalries > Iowa: ...`. That label does a lot of work on this page. The rivalry paragraphs keep saying "the teams" and "the series" without naming the opponent, so without it a chunk about Iowa looks a lot like one about Kansas or Missouri. The label only goes into the embedding. The model sees the section name separately, in the prompt.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator="end",
)

chunks = []
for sec in sections:
    for piece in splitter.split_text(sec["text"]):
        chunks.append({
            "id": len(chunks),
            "section": sec["section"],
            "text": piece,
            "embed_text": f"{page_title} > {sec['section']}: {piece}",
        })

lengths = [len(c["text"]) for c in chunks]
print(f"{len(chunks)} chunks | length min/median/max: {min(lengths)} / {int(np.median(lengths))} / {max(lengths)} chars")
pd.DataFrame(chunks)[["id", "section", "text"]].head(8)

79 chunks | length min/median/max: 68 / 587 / 797 chars


,id,section,text
0,0,Introduction,The Nebraska Cornhuskers football team compete...
1,1,Introduction,The program's first extended period of success...
2,2,Introduction,"Following Osborne's retirement in 1997, Nebras..."
3,3,Conference affiliations,"- Independent (1890–1891, 1898–1906, 1919–1920..."
4,4,Head coaches,Nebraska has had thirty-one head coaches in it...
5,5,Head coaches,"NU's earliest coaching history is unclear, as ..."
6,6,Head coaches,Dana X. Bible and Biff Jones guided Nebraska t...
7,7,Head coaches,Despite remarkable consistency – Osborne's tea...


## Step 3: Embeddings and the vector index

The embedding model is `sentence-transformers/all-MiniLM-L6-v2`. The model is approximately 80 MB and decently fast even without the Colab GPU. It was also trained for the exact job we're doing which is judging whether two short pieces of text mean similar things. Something bigger like `all-mpnet-base-v2` or `BAAI/bge-base-en-v1.5` would most likely be better at retrieving though. Despite this, for 79 chunks the small one is plenty acceptable.

The vectors are normalized, so an inner-product search in FAISS gives cosine similarity. A score near 1 means very similar and near 0 means unrelated.

The index is flat, meaning brute force. Approximate indexes only make sense with hundreds of thousands of vectors. With 79, exact search takes no time at all.

In [6]:
embedder = SentenceTransformer(EMBED_MODEL, device=device)

chunk_vectors = embedder.encode(
    [c["embed_text"] for c in chunks],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

index = faiss.IndexFlatIP(chunk_vectors.shape[1])
index.add(chunk_vectors)
print(f"Indexed {index.ntotal} vectors of dimension {chunk_vectors.shape[1]}")


def retrieve(query, k=TOP_K):
    query_vector = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(query_vector, k)
    return [{**chunks[i], "score": float(s)} for s, i in zip(scores[0], ids[0]) if i != -1]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Indexed 79 vectors of dimension 384


Before bringing in the LLM, retrieval gets tested on its own. If the right chunk isn't in the top 4, nothing the prompt does later can save the answer.

In [7]:
def show_retrieval(query, k=TOP_K):
    print(f"Q: {query}")
    for rank, hit in enumerate(retrieve(query, k), start=1):
        snippet = hit["text"][:160].replace("\n", " ")
        print(f"  {rank}. score={hit['score']:.3f}  [{hit['section']}]  {snippet}...")
    print()

for q in ["How many national championships has Nebraska won?",
          "What are the Blackshirts?",
          "What trophy do Nebraska and Iowa play for?"]:
    show_retrieval(q)

Q: How many national championships has Nebraska won?
  1. score=0.757  [Championships > National championships]  Nebraska claims five national championships, each awarded by the AP poll or the Coaches poll....
  2. score=0.754  [Championships > National championships]  Nebraska has been awarded seven other national championships from NCAA-designated major selectors the school does not claim....
  3. score=0.707  [Championships > National championships]  Nebraska's first titles came less than a decade into Bob Devaney's tenure. Devaney used a quarterback rotation between Jerry Tagge and Van Brownson throughout 1...
  4. score=0.704  [Championships > National championships]  NU still trailed Michigan in both polls, but its Orange Bowl matchup with Tennessee was the designated Bowl Alliance national championship game. The Cornhuskers...

Q: What are the Blackshirts?
  1. score=0.416  [Traditions > Uniform history]  Nebraska's defense is often referred to as the "Blackshirts," a reference 

### Commentary

Initial analysis shows that the retrieval got the correct material but the scores state differently.

For the national championships question, all four hits came from the same section and the top two were nearly tied (0.757 and 0.754). The second hit is the paragraph about the seven titles Nebraska doesn't claim. This is an intentional trap and a model reading both could say five, seven or twelve.

The Blackshirts question is by far the best result. The right chunk scored 0.416 and the next best was 0.248. The top score is low compared to the other questions, though, which shows that similarity scores aren't comparable across questions. A short question built around one unusual word just doesn't produce high numbers, even when the match is exact.

The Iowa question put the right section in the top two spots, and the first hit contains "Heroes Trophy." The third hit was the Wisconsin rivalry at 0.659, only 0.06 behind. That paragraph mentions a different trophy, so it's exactly the kind of look-alike rivalry chunk I expected to cause trouble. The fourth hit, Conference affiliations, is just a list of conferences and is pure noise.

My takeaway is that a fixed top 4 is blunt. For narrow questions it pads the prompt with distractors, and for broad ones 4 chunks may not be enough.

## Step 4: The language model

The generator is `Qwen/Qwen2.5-1.5B-Instruct`. Two things decided it. First, it's instruction-tuned with a chat template. That's non-negotiable here, because the whole setup depends on the model obeying "only use the context below," and a base model would just keep writing. Second, it fits on a free T4 in half precision (about 3 GB of weights) and averaged 1.9 seconds per answer in the Step 6 run. A 3B or 7B model would write better answers but would be slow or wouldn't fit on the free tier at all.

Decoding is greedy (`do_sample=False`). For factual answers, randomness just adds ways to be wrong, and it would make the evaluation impossible to repeat.

The model is loaded directly with the tokenizer and `generate` instead of through `pipeline`. That keeps the chat template visible and makes it easy to cut the prompt off the front of the output.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
llm.eval()


def generate(messages, max_new_tokens=300):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(llm.device)
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## Step 5: The RAG chat function

The system prompt has three jobs. It keeps the model on the excerpts and away from its own memory. It gives the model a way out, an exact sentence to use when the answer isn't there. Small models are trained to be helpful, and without a permitted "I don't know" they tend to make something up. And it asks for the section name, so a reader can check where the answer came from.

Each retrieved chunk goes into the prompt numbered and labeled with its section, so the model can see what it's citing.

Turning this into a chat adds a wrinkle. A follow-up like "How many titles did he win?" causes trouble in two places. The model needs to know who "he" is, so the last couple of exchanges get passed in as chat history. Retrieval has the same problem, and it's easier to miss: embed "How many titles did he win?" by itself and the result is close to meaningless. The cheap fix used here glues the previous question onto the new one before searching. A better fix would have the model rewrite the follow-up as a standalone question first, at the cost of an extra model call every turn.

History only keeps the plain questions and answers, not the retrieved excerpts, so the prompt doesn't balloon as the conversation goes on.

Setting `use_rag=False` skips retrieval entirely. That's the baseline in Step 6: the same model answering from memory.

In [9]:
SYSTEM_PROMPT = f'''You are a helpful assistant that answers questions about the Wikipedia article "{page_title}".
Answer ONLY using the context excerpts provided with each question. Do not use outside knowledge.
If the excerpts do not contain the answer, reply exactly: "I couldn't find that in the article."
Keep answers concise (2-4 sentences) and mention which section(s) the information came from.'''

BASELINE_PROMPT = "You are a helpful assistant. Answer the question concisely."


def answer(question, history=None, k=TOP_K, use_rag=True):
    history = history or []

    if use_rag:
        previous_questions = [m["content"] for m in history if m["role"] == "user"][-1:]
        retrieval_query = " ".join(previous_questions + [question])
        hits = retrieve(retrieval_query, k)
        context = "\n\n".join(
            f"[{i}] (Section: {h['section']})\n{h['text']}" for i, h in enumerate(hits, start=1)
        )
        user_message = f"Context excerpts:\n{context}\n\nQuestion: {question}"
        system = SYSTEM_PROMPT
    else:
        hits = []
        user_message = question
        system = BASELINE_PROMPT

    messages = [{"role": "system", "content": system}] + history[-4:] + [
        {"role": "user", "content": user_message}
    ]
    return generate(messages), hits


reply, hits = answer("How did Bob Devaney change the program?")
print(textwrap.fill(reply, 100))
print("\nRetrieved from:", [h["section"] for h in hits])

Under Bob Devaney's leadership, Nebraska transformed itself into a national powerhouse. He led the
team to two national championships and eight conference titles within just eleven seasons as head
coach. His innovative strategies, including the I formation offense and groundbreaking strength,
conditioning, and nutrition programs, significantly improved the team's performance and set new
standards for college football coaching.

Retrieved from: ['Traditions > Walk-on program', 'Head coaches', 'Introduction', 'Sellout streak']


## Step 6: Evaluation

Asking a few questions and eyeballing the answers isn't really an evaluation, so this is a small structured test. Every question has a known answer in the article, and three things get scored separately:

| Measure | What it checks |
|---|---|
| Retrieval hit | Did any of the top 4 chunks contain the answer? |
| RAG correct | Did the chatbot's answer contain the right fact? |
| Baseline correct | Did the same model get it right with no retrieval? |

Scoring retrieval apart from the answer shows where something broke. If retrieval missed, the fix belongs in chunking or embeddings. If retrieval found it and the answer was still wrong, the problem is the prompt or the model.

The last two questions shouldn't be answerable. The Titanic one is unrelated on purpose. The volleyball coach question is the sneaky one. It's about Nebraska, and the article even mentions a volleyball event at Memorial Stadium, but it never names the coach. The right answer to both is "it's not in the article," even if the model thinks it knows.

Two questions are scored differently. The Heisman question only counts if all three winners are named. The sellout question accepts either 411 or 415, because the article contradicts itself: the Memorial Stadium section says 415 and the Sellout streak section says 411 (as of October 2026). Which one the chatbot says depends on which chunk it pulls.

The scoring is plain keyword matching, which is crude. An answer saying "not 1923" would count as correct for the stadium question, and a right answer worded differently from the keyword can get marked wrong. With 12 questions it's easy to read every answer, so the full answers print below the table.

The expected answers come from the article as it read in October 2026. Wikipedia changes, so check them against the live page if this gets rerun later.

In [10]:
REFUSAL = ["couldn't find", "could not find", "not in the article", "does not", "doesn't",
           "not mention", "no information"]

eval_set = [
    {"question": "Which coach led Nebraska to its 1970 and 1971 national championships?",
     "expected": ["Devaney"]},
    {"question": "Who was Nebraska's first official head coach?",
     "expected": ["Crawford"]},
    {"question": "What year did Memorial Stadium open?",
     "expected": ["1923"]},
    {"question": "Which Nebraska players have won the Heisman Trophy?",
     "expected": ["Rodgers", "Rozier", "Crouch"], "require_all": True},
    {"question": "Why is Nebraska's defense called the Blackshirts?",
     "expected": ["practice"]},
    {"question": "What song is played during the Tunnel Walk?",
     "expected": ["Sirius"]},
    {"question": "What trophy do Nebraska and Iowa play for?",
     "expected": ["Heroes"]},
    {"question": "What was the final score of the 1971 Game of the Century?",
     "expected": ["35–31", "35-31", "35 to 31"]},
    {"question": "How many consecutive seasons did Nebraska play in a bowl game during its record streak?",
     "expected": ["35", "thirty-five"]},
    {"question": "How many consecutive sellouts does Nebraska have at Memorial Stadium?",
     "expected": ["411", "415"]},
    {"question": "What year did the Titanic sink?",
     "expected": REFUSAL, "out_of_scope": True},
    {"question": "Who is the head coach of Nebraska's volleyball team?",
     "expected": REFUSAL, "out_of_scope": True},
]


def contains_any(text, keywords, require_all=False):
    text = text.lower()
    found = [k.lower() in text for k in keywords]
    return all(found) if require_all else any(found)


rows = []
for item in eval_set:
    out_of_scope = item.get("out_of_scope", False)
    need_all = item.get("require_all", False)

    start = time.time()
    rag_answer, hits = answer(item["question"])
    rag_seconds = time.time() - start
    baseline_answer, _ = answer(item["question"], use_rag=False)

    retrieved_text = " ".join(h["text"] for h in hits)
    rows.append({
        "question": item["question"],
        "retrieval_hit": None if out_of_scope else contains_any(retrieved_text, item["expected"], need_all),
        "rag_correct": contains_any(rag_answer, item["expected"], need_all),
        "baseline_correct": None if out_of_scope else contains_any(baseline_answer, item["expected"], need_all),
        "top_score": round(hits[0]["score"], 3) if hits else None,
        "rag_seconds": round(rag_seconds, 1),
        "rag_answer": rag_answer,
        "baseline_answer": baseline_answer,
    })

results = pd.DataFrame(rows)
results[["question", "retrieval_hit", "rag_correct", "baseline_correct", "top_score", "rag_seconds"]]

,question,retrieval_hit,rag_correct,baseline_correct,top_score,rag_seconds
0,Which coach led Nebraska to its 1970 and 1971 ...,True,True,False,0.746,2.2
1,Who was Nebraska's first official head coach?,True,True,False,0.730,1.1
2,What year did Memorial Stadium open?,True,True,False,0.454,1.3
3,Which Nebraska players have won the Heisman Tr...,True,True,False,0.666,3.6
4,Why is Nebraska's defense called the Blackshirts?,True,True,False,0.769,3.5
5,What song is played during the Tunnel Walk?,True,True,False,0.361,4.7
6,What trophy do Nebraska and Iowa play for?,True,True,False,0.723,1.6
7,What was the final score of the 1971 Game of t...,True,False,False,0.476,1.2
8,How many consecutive seasons did Nebraska play...,True,True,False,0.688,1.2
9,How many consecutive sellouts does Nebraska ha...,True,True,False,0.720,1.8


In [11]:
in_scope = results[results["retrieval_hit"].notna()]
print(f"Retrieval hit rate : {in_scope['retrieval_hit'].mean():.0%}")
print(f"RAG accuracy       : {in_scope['rag_correct'].mean():.0%}")
print(f"Baseline accuracy  : {in_scope['baseline_correct'].astype(bool).mean():.0%}")
out_of_scope = results[results["retrieval_hit"].isna()]
print(f"Out-of-scope questions refused: {int(out_of_scope['rag_correct'].sum())} of {len(out_of_scope)}")
print(f"Average RAG latency: {results['rag_seconds'].mean():.1f} s\n")

for _, row in results.iterrows():
    print("=" * 100)
    print("Q:", row["question"])
    print("\nRAG     :", textwrap.fill(row["rag_answer"], 92, subsequent_indent=" " * 10))
    print("\nBASELINE:", textwrap.fill(row["baseline_answer"], 92, subsequent_indent=" " * 10))

Retrieval hit rate : 100%
RAG accuracy       : 90%
Baseline accuracy  : 0%
Out-of-scope questions refused: 2 of 2
Average RAG latency: 1.9 s

Q: Which coach led Nebraska to its 1970 and 1971 national championships?

RAG     : Bob Devaney led Nebraska to its 1970 and 1971 national championships.

BASELINE: Tom Osborne coached Nebraska to its two national championships in 1970 and 1971.
Q: Who was Nebraska's first official head coach?

RAG     : Frank Crawford was Nebraska's first official head coach according to the provided excerpt.

BASELINE: Nebraska's first official head football coach was Frank H. Vandiver, who served from 1904 to
          1923.
Q: What year did Memorial Stadium open?

RAG     : Memorial Stadium opened on October 13, 1923.

BASELINE: Memorial Stadium opened in 1925.
Q: Which Nebraska players have won the Heisman Trophy?

RAG     : Johnny Rodgers (1972), Mike Rozier (1983), and Eric Crouch (2001).

BASELINE: The only Nebraska player to win the Heisman Trophy is Mik

### Commentary

The gap between RAG and the baseline is the clearest result in the notebook. With retrieval the model got 9 of 10 answerable questions by the keyword check. Without it, 0 of 10. And the baseline didn't fail by saying "I don't know." It failed confidently. It said Nebraska's only Heisman winner was Mike Leach in 1987 (Nebraska has three, and none of them is Mike Leach). It gave Tom Osborne the 1970 and 1971 titles, put Memorial Stadium's opening in 1925, called the Game of the Century "Super Bowl III," and named Tom Allen as the volleyball coach. Every one of those sounds plausible to someone who doesn't know better. That's the real case for RAG with a small model. It doesn't just add facts, it keeps the model from inventing them.

I expected the baseline to do better on famous facts than obscure ones. It didn't. It missed Devaney and the Heisman winners just as badly as the Tunnel Walk song. A 1.5B model simply doesn't hold much specific knowledge, famous or not.

The one RAG answer marked wrong, the Game of the Century, is a scoring error. The model said "Nebraska 35, Oklahoma 31," which is correct; the check was looking for "35-31" written as one string. Counting it, RAG went 10 for 10. But the bowl streak answer shows that passing the keyword check isn't the same as being fully right. The model correctly said 35 straight bowl games, then added that Nebraska went 27-28 during the streak. 27-28 is Nebraska's all-time bowl record across 55 games, not its record during the streak. It glued two numbers from the same chunk together wrong, and the keyword check can't see that.

The same kind of mistake shows up in the Step 5 demo. Asked how Bob Devaney changed the program, the model credited him with the I formation offense and the strength, conditioning and nutrition programs. The introduction gives those to Tom Osborne. Both coaches are in the same paragraph and the model blended them. Retrieval brought the right text and the model misread it, so that's a generation failure. A bigger model, or a prompt that makes it quote its evidence, would be the place to fix it.

Both out-of-scope questions were refused with the exact sentence from the prompt. I thought the volleyball question might slip through, since it's about Nebraska and the article mentions volleyball, but it held. Two questions is a small sample, though.

On the sellout streak, the chatbot said 415, from the Memorial Stadium section, and never mentioned that the Sellout streak section says 411. The answer is "right" according to the article, but the user would never know the source disagrees with itself. RAG passes along whatever the source says, contradictions included.

Citations were the weakest part. None of the ten answers named a section, even though the prompt asked for it. One just said "according to the provided excerpt." The model follows "only use the context" much better than "say where it came from." If sources matter, the code should attach them from the retrieved hits, which the chat interface in Step 8 already does, instead of trusting the model to do it.

Speed is fine for a chat. The average answer took 1.9 seconds and the slowest took 4.7.

## Step 7: Multi-turn conversation

A short scripted conversation to see how follow-ups behave. The second and third questions use "he" and don't make sense alone, so they only work if both retrieval and the model keep track of the earlier turns.

In [12]:
history = []
for question in ["Who succeeded Bob Devaney as head coach?",
                 "How many national championships did he win?",
                 "Who took over after he retired?"]:
    reply, hits = answer(question, history)
    history += [{"role": "user", "content": question},
                {"role": "assistant", "content": reply}]
    print("You:", question)
    print("Bot:", textwrap.fill(reply, 95, subsequent_indent="     "))
    print("     sources:", ", ".join(dict.fromkeys(h["section"] for h in hits)), "\n")

You: Who succeeded Bob Devaney as head coach?
Bot: Tom Osborne succeeded Bob Devaney as head coach.
     sources: Head coaches, Introduction, Championships > Conference championships 

You: How many national championships did he win?
Bot: Bob Devaney won two national championships.
     sources: Head coaches, Introduction, Championships > Conference championships 

You: Who took over after he retired?
Bot: After Bob Devaney retired in 1972, Tom Osborne took over as head coach.
     sources: Championships > National championships, Rivalries > Oklahoma, Sellout streak 



### Commentary

The follow-ups went wrong, and the way they went wrong is useful. Turn 1 was fine: Tom Osborne succeeded Devaney. In turn 2, "he" should mean Osborne, but the model answered about Devaney ("two national championships"). Turn 3 built on that mistake. It treated "he" as Devaney again and said Osborne took over, when the right answer, the coach after Osborne retired, is Frank Solich.

The cause is my retrieval shortcut. It glues the previous question onto the new one, and the previous question was "Who succeeded Bob Devaney as head coach?" Osborne's name only appears in the previous answer, which the search never sees. So the retrieval query was all about Devaney, and the only person named in the question history was Devaney. The model went with him.

Rewriting the follow-up into a standalone question before searching, the fix I mentioned in Step 5, should catch this. An even cheaper fix would be to include the last answer in the retrieval query, not just the last question. Either way, this is the most serious weakness in the chatbot, because real users ask follow-ups constantly.

## Step 8: Chat interface

This puts `answer()` behind a Gradio chat window like the one in the reading. Each reply ends with the sections it drew from.

Gradio has changed how it passes chat history between versions. Older releases send a list of `(user, bot)` tuples and newer ones send `{"role", "content"}` dicts, so the code handles both. It also strips the "Sources" line off earlier replies before they go back to the model, since that line isn't part of the actual answer.

In [13]:
import gradio as gr


def to_messages(history):
    messages = []
    for turn in history or []:
        if isinstance(turn, dict):
            if isinstance(turn.get("content"), str):
                messages.append({"role": turn["role"], "content": turn["content"].split("\n\n_Sources:")[0]})
        else:
            user_text, bot_text = turn
            messages.append({"role": "user", "content": user_text})
            if bot_text:
                messages.append({"role": "assistant", "content": bot_text.split("\n\n_Sources:")[0]})
    return messages


def chat_fn(message, history):
    reply, hits = answer(message, to_messages(history))
    sources = ", ".join(dict.fromkeys(h["section"] for h in hits))
    return f"{reply}\n\n_Sources: {sources}_"


demo = gr.ChatInterface(
    fn=chat_fn,
    title=f"Chat with Wikipedia: {page_title}",
    description=f"Answers are generated only from the Wikipedia article {page_url}",
    examples=["What is the Tunnel Walk?",
              "Tell me about the Game of the Century.",
              "How long is the sellout streak?"],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: (temporary share link removed)

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


If the Gradio window doesn't show up, this text loop does the same thing. Type `quit` to stop.

In [14]:
history = []
while True:
    question = input("You: ").strip()
    if question.lower() in {"", "quit", "exit"}:
        break
    reply, hits = answer(question, history)
    history += [{"role": "user", "content": question},
                {"role": "assistant", "content": reply}]
    print("Bot:", textwrap.fill(reply, 95, subsequent_indent="     "))
    print("     sources:", ", ".join(dict.fromkeys(h["section"] for h in hits)), "\n")

You: quit


## Conclusions and reflection

The chatbot works well for single factual questions about the article. It answered 9 of 10 correctly by the strict keyword check (10 of 10 counting the scoring miss), refused both questions the article doesn't cover, and answered in about two seconds. I'd trust it for a quick lookup as long as the sources are shown. I wouldn't trust it in a conversation yet. The follow-up test showed it losing track of who "he" is within two turns, and it sometimes blends facts that sit in the same paragraph.

The assignment says the input is just text no matter where it comes from. Once the text was clean, that held up, and nothing after Step 1 needed to know the text came from Wikipedia. Getting clean text out of the HTML was the hardest part of the build, though. The first version of the scraper returned zero sections because of a reused class name, and the cleaning rules had to strip references, edit links and tables that a PDF loader would never have produced. HTML also gave something a PDF usually doesn't: real section headings. Those ended up being the most useful thing in the pipeline.

That's the biggest lesson for me. The section labels did more than any other single choice. They help retrieval tell look-alike paragraphs apart, they let the chat interface show sources, and they make failures easy to trace. A small change to the label, switching its separator to ">", flipped the bowl streak question from a retrieval miss to a hit. That's one run, so I wouldn't lean on it too hard, but it shows how sensitive a small embedding model is to the exact text it's given.

If I kept working on this, I'd do three things, in this order. First, rewrite follow-up questions into standalone ones before retrieval, since the conversation failure is the most serious problem. Second, add keyword search alongside the embeddings, because so many questions here hinge on exact names and numbers (Heroes Trophy, Sirius, 35-31). Third, try a 3B model to see whether it stops mixing up Devaney and Osborne. I'd hold off on a bigger embedding model. In the final run, retrieval found the answer for every answerable question, so it isn't the bottleneck right now.